# 47 — Pool-native qualitative gallery blocks (+ λ contrast)

Fills the two pending TODO items of `plans/qualitative-gallery.md`:

1. **Two lead blocks built from validated-pool concepts** (nb32 pool), same
   format as the existing gallery: nested quad, n = 1 → 4 progression, simple
   blend, greedy Top-k (k = 4, 32 steps, λ = 0). One block feeds the report's
   §5.5 inline-example slot.
2. **One λ = 0 vs λ = 0.4 contrast pair** on the lead block at n = 2, so the
   gallery also illustrates the dial the report recommends.

Quads (all pool members, chosen for interpretability and narrative register;
swap freely — the full pool is printed below):

- **novel** — sorrow → fury → flood → passing ("Write the opening of a novel.")
- **traveler** — country → town → dwelling → respite ("The traveler arrived at")

Concepts are built under the same policy as every quantitative Llama run
(8 languages, min_lemmas = 11). Output: `resources/47_qualitative_pool.jsonl`
plus a formatted-markdown print for pasting into the gallery.

In [ ]:
import json
from pathlib import Path

import torch

from frames.evaluation.harness import EvaluationHarness
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept
from frames.utils.memory import gc_cuda

In [ ]:
MAX_TOKENS = 3
K = 4
STEPS = 32  # gallery setting (the quantitative runs use 24)
MIN_LEMMAS = 11  # Llama policy, matching all quantitative runs
LAMBDAS = (0.0, 0.4)  # contrast pair, lead block only

POOL = json.loads(Path("resources/32_concept_pool_bf16.json").read_text())
POOL_DELTA = {e["synset"]: round(e["delta"], 2) for e in POOL["pool"]}
print("pool members:", ", ".join(sorted(POOL_DELTA)))

QUADS = {
    "novel": ["sorrow.n.01", "fury.n.01", "flood.n.01", "passing.n.02"],
    "traveler": ["country.n.02", "town.n.02", "dwelling.n.01", "respite.n.04"],
}
assert all(m in POOL_DELTA for q in QUADS.values() for m in q)


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


PROMPTS = {
    "novel": chat("Write the opening of a novel.", ""),
    "traveler": chat("Tell me a short story.", "The traveler arrived at"),
}

LOG_PATH = Path("resources/47_qualitative_pool.jsonl")
LOG_PATH.unlink(missing_ok=True)
BASELINES_PATH = Path("resources/47_baselines.json")

In [ ]:
with gc_cuda():
    fur = FrameUnembeddingRepresentation.from_model_id(
        id="meta-llama/Llama-3.1-8B-Instruct",
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    harness = EvaluationHarness(
        fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
    )

    concept = {}
    for name in sorted({m for q in QUADS.values() for m in q}):
        c = fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
        concept[name] = Concept(synset=c.synset, tensor=c.tensor.to(fur.model.device))

    baselines = {}
    for block, quad in QUADS.items():
        prompt = [PROMPTS[block]]
        baseline_texts = harness.generate_baseline(prompt, max_new_tokens=STEPS + 2)
        baselines[block] = baseline_texts[0]

        # n = 1..4 progression, canonical greedy loop (lambda = 0)
        for n in range(1, 5):
            names = quad[:n]
            guide = (
                concept[names[0]]
                if n == 1
                else Concept.average([concept[m] for m in names])
            )
            torch.cuda.empty_cache()
            texts, probe = fur.generate_with_topk_guide(
                prompt, guide=guide, k=K, steps=STEPS
            )
            harness.evaluate(
                prompt,
                texts,
                names,
                config={"block": block, "n": n, "lam": 0.0, "kind": "progression"},
                probe=probe,
                baseline_texts=baseline_texts,
                concepts={m: concept[m] for m in names},
                angles=True,
            )

        # lambda contrast on the lead block at n = 2 (both values via the same
        # multi-guide path so the pair is internally comparable)
        if block == "novel":
            names = quad[:2]
            blend = Concept.average([concept[m] for m in names])
            for lam in LAMBDAS:
                torch.cuda.empty_cache()
                texts, probe = fur.generate_with_topk_multi_guide(
                    prompt, guides=[blend], k=K, steps=STEPS, fluency_weight=lam
                )
                harness.evaluate(
                    prompt,
                    texts,
                    names,
                    config={"block": block, "n": 2, "lam": lam, "kind": "contrast"},
                    probe=probe,
                    baseline_texts=baseline_texts,
                    concepts={m: concept[m] for m in names},
                    angles=True,
                )

    BASELINES_PATH.write_text(json.dumps(baselines, indent=1))
    print("done")

In [ ]:
records = [json.loads(x) for x in LOG_PATH.read_text().strip().split("\n")]
baselines = json.loads(BASELINES_PATH.read_text())


def short(name):
    return name.split(".")[0]


def fmt(r):
    c = r["config"]
    names = [n for n in QUADS[c["block"]][: c["n"]]]
    deltas = {n: r["expression"][n]["delta"] for n in names}
    ok = min(deltas.values()) > 0
    dstr = ", ".join(f"{short(n)} {d:+.2f}" for n, d in deltas.items())
    mark = "✓ all improved" if ok else "✗ partial"
    lam = f", λ={c['lam']}" if c["kind"] == "contrast" else ""
    text = r["continuation"]
    return (
        f"**n = {c['n']}: {' + '.join(short(n) for n in names)}"
        f"{lam}** — Δ [{dstr}] — {mark}\n> {text}\n"
    )


for block, quad in QUADS.items():
    arrow = " → ".join(short(m) for m in quad)
    pool_line = " · ".join(f"{short(m)} {POOL_DELTA[m]}" for m in quad)
    print(f"## Block P{list(QUADS).index(block) + 1} — pool-native: {block}\n")
    print(f"*Quad: {arrow} — all validated pool members.*")
    print(f"*Validated singles Δ (nb32 pool): {pool_line}.*\n")
    print(f"> **Unsteered baseline.** {baselines[block]}\n")
    for r in records:
        if r["config"]["block"] == block and r["config"]["kind"] == "progression":
            print(fmt(r))
    contrast = [
        r
        for r in records
        if r["config"]["block"] == block and r["config"]["kind"] == "contrast"
    ]
    if contrast:
        print("**λ contrast (n = 2):**\n")
        for r in contrast:
            print(fmt(r))
    print("---\n")